In [ ]:
%cd ..

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import pandas as pd
import zarr
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from common.sample_db import SampleDB

db = SampleDB()

In [ ]:
losses = pd.read_csv('/dev/shm/sobol.losses.txt', header=None, names=['loss'])
losses.head()

### distributiuon of losses from score

In [ ]:
losses = db.losses_for(run='004', model='231_keras/i0')
losses_1_df = pd.DataFrame(losses)
losses_1_df['src'] = 'sobol'
losses_1_df.describe()

In [ ]:
losses = []
for run in range(200, 209):
    run = f"{run:03d}"
    losses.extend(db.losses_for(run=run, model='231_keras/i0'))
losses_2_df = pd.DataFrame(losses)
losses_2_df['src'] = 'hard_mined'
losses_2_df.describe()

In [ ]:
losses_df = pd.concat([losses_1_df, losses_2_df], ignore_index=True)
#losses_df = pd.concat([losses_1_df], ignore_index=True)
losses_df['stft'] *= 0.01
losses_df.tail()

In [ ]:
src_values = losses_df['src'].unique()
cols = ['huber', 'stft']
shared_xlim = (
    losses_df[cols].min().min(),
    losses_df[cols].max().max(),
)

fig, axes = plt.subplots(len(src_values), len(cols), figsize=(12, 4 * len(src_values)))

for row, src in enumerate(src_values):
    subset = losses_df[losses_df['src'] == src]
    for col_idx, col in enumerate(cols):
        ax = axes[row, col_idx]
        sns.histplot(data=subset, x=col, kde=True, stat="density", alpha=0.5, ax=ax)
        ax.set_xlim(shared_xlim)
        ax.set_title(f"{col} — {src}")

plt.tight_layout()
plt.show()
